In [1]:
# ── CELL 1b: GPU Check ──────────────────────────────────
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    vram_gb = gpu.total_memory / 1e9
    print(f"✅ GPU detected: {gpu.name}")
    print(f"   Total VRAM : {vram_gb:.1f} GB")
    if vram_gb >= 35:
        print("   → A100/H100. All 10 models run in full BF16.")
    elif vram_gb >= 14:
        print("   → T4. Borderline models will use 8-bit quantization.")
    else:
        print(f"   ⚠️ Only {vram_gb:.1f} GB — request a higher runtime tier.")
else:
    print("❌ No GPU! Go to Runtime → Change runtime type → GPU first.")


✅ GPU detected: NVIDIA GeForce RTX 3050 6GB Laptop GPU
   Total VRAM : 6.4 GB
   ⚠️ Only 6.4 GB — request a higher runtime tier.


# VLM Bias Prediction — 10-Model Colab Evaluation Pipeline

**Instructions:**
1. Upload `counterfactual_images/` and `prior_strength_scores.csv` to Google Drive under `MyDrive/vlm_bias_prediction/`
2. Add your `GROQ_API_KEY` and `HF_TOKEN` in Colab **Secrets** (🔑 icon in left sidebar)
3. Run cells from top to bottom. Each model section is **independent** — if one crashes, the next still works.
4. Results are saved to Drive after every image. Safe to interrupt and resume.

In [6]:
# ── CELL 1: Install Dependencies ──────────────────────────────────────────────
!pip install -q transformers>=4.45.0 accelerate bitsandbytes qwen-vl-utils pillow pandas groq
#!pip install flash-attn --no-build-isolation
!pip install -q \
    transformers>=4.45.0 \
    accelerate \
    bitsandbytes \
    qwen-vl-utils \
    pillow \
    pandas \
    groq \
    num2words \
    torchvision \
    einops \
    sentencepiece \
    protobuf
!pip install qwen-vl-utils

In [2]:
# ── CELL 2: Mount Google Drive & Set Paths ────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [3]:


import os
DRIVE_BASE   = "/content/drive/MyDrive/MS Research/PE/data/raw"
# DRIVE_BASE   = "/content/drive/MyDrive/MS Research/data/raw"
IMAGE_DIR    = os.path.join(DRIVE_BASE, "counterfactual_images")
RESULTS_DIR  = os.path.join(DRIVE_BASE, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Images found: {len(os.listdir(IMAGE_DIR))}")
print(f"Results will be saved to: {RESULTS_DIR}")

Images found: 60
Results will be saved to: /content/drive/MyDrive/MS Research/PE/data/raw/results


In [15]:
# ── CELL 3: Load API Keys & Shared Utilities ──────────────────────────────────
import re, base64, torch, pandas as pd
from pathlib import Path
from PIL import Image
import os
os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN_HERE"
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY_HERE"
PROMPT = "How many legs does this animal exactly have in this image? Answer with just a single number integer, nothing else."

def parse_legs(text):
    """Extract integer prediction from model text response."""
    if not text: return None
    matches = re.findall(r'\b(\d+)\b', text.strip())
    return int(matches[0]) if matches else None

def parse_filename(fname):
    """Extract concept, true_legs, background from filename like 'cat_3legs_natural.png'."""
    name = Path(fname).stem
    m = re.match(r'^(.+?)_(\d+)legs_(natural|plain)$', name)
    if m:
        return m.group(1), int(m.group(2)), m.group(3)
    return name, None, None

def get_all_images():
    files = sorted([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(('.png','.jpg','.jpeg'))])
    return files

def save_results(records, csv_path):
    df = pd.DataFrame(records)
    df.to_csv(csv_path, index=False)

def free_gpu():
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    print(f"GPU freed. Current VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

print(f"VRAM available: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("Utilities loaded. Ready to evaluate.")

VRAM available: 15.6 GB
Utilities loaded. Ready to evaluate.


In [16]:
HF_TOKEN     = os.environ["HF_TOKEN"]
GROQ_API_KEY = os.environ["GROQ_API_KEY"]

---
## Model 1: SmolVLM2-2.2B (Smoke Test — Run First)
**Why first:** Smallest model (5GB VRAM). Validates the full pipeline before committing time to larger models.

In [9]:
!pip install -q --upgrade transformers>=4.45.0
import importlib, transformers
importlib.reload(transformers)
print("transformers version:", transformers.__version__)

transformers version: 5.3.0


In [6]:
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID  = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"
CSV_OUT   = os.path.join(RESULTS_DIR, "smolvlm2_results.csv")

processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
model     = AutoModelForImageTextToText.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto", token=HF_TOKEN)

records = list(pd.read_csv(CSV_OUT).to_dict('records')) if os.path.exists(CSV_OUT) else []
done    = {r['image'] for r in records}

for fname in get_all_images():
    if fname in done: print(f"  skip {fname}"); continue
    concept, true_legs, bg = parse_filename(fname)
    img = Image.open(os.path.join(IMAGE_DIR, fname)).convert('RGB')

    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": PROMPT}]}]
    prompt_text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=prompt_text, images=[img], return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=10)
    text = processor.decode(out[0], skip_special_tokens=True)
    pred = parse_legs(text)
    is_biased = int(pred != true_legs) if pred is not None else 1

    records.append({"image": fname, "concept": concept, "true_legs": true_legs,
                    "background": bg, "predicted": pred, "is_biased": is_biased,
                    "model": "SmolVLM2-2.2B", "hf_id": MODEL_ID})
    save_results(records, CSV_OUT)
    print(f"SmolVLM2 | {fname} | pred={pred} | true={true_legs} | bias={is_biased}")

del model, processor; free_gpu()
print(f"\n✅ SmolVLM2 done. Results: {CSV_OUT}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/657 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

SmolVLM2 | binturong_3legs_natural.png | pred=4 | true=3 | bias=1
SmolVLM2 | binturong_3legs_plain.png | pred=4 | true=3 | bias=1
SmolVLM2 | binturong_5legs_natural.png | pred=4 | true=5 | bias=1
SmolVLM2 | binturong_5legs_plain.png | pred=4 | true=5 | bias=1
SmolVLM2 | cat_3legs_natural.png | pred=4 | true=3 | bias=1
SmolVLM2 | cat_3legs_plain.png | pred=4 | true=3 | bias=1
SmolVLM2 | cat_5legs_natural.png | pred=4 | true=5 | bias=1
SmolVLM2 | cat_5legs_plain.png | pred=4 | true=5 | bias=1
SmolVLM2 | cheetah_3legs_natural.png | pred=4 | true=3 | bias=1
SmolVLM2 | cheetah_3legs_plain.png | pred=4 | true=3 | bias=1
SmolVLM2 | cheetah_5legs_natural.jpg | pred=4 | true=5 | bias=1
SmolVLM2 | cheetah_5legs_plain.png | pred=4 | true=5 | bias=1
SmolVLM2 | deer_3legs_natural.png | pred=4 | true=3 | bias=1
SmolVLM2 | deer_3legs_plain.png | pred=4 | true=3 | bias=1
SmolVLM2 | deer_5legs_natural.jpg | pred=4 | true=5 | bias=1
SmolVLM2 | deer_5legs_plain.png | pred=4 | true=5 | bias=1
SmolVLM2 | d

---
## Model 2: LLaVA-1.5-7B (Legacy Baseline)

In [8]:
from transformers import LlavaForConditionalGeneration, AutoProcessor

MODEL_ID = "llava-hf/llava-1.5-7b-hf"
CSV_OUT  = os.path.join(RESULTS_DIR, "llava15_results.csv")

processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
model     = LlavaForConditionalGeneration.from_pretrained(MODEL_ID,
                device_map="auto", token=HF_TOKEN)

records = list(pd.read_csv(CSV_OUT).to_dict('records')) if os.path.exists(CSV_OUT) else []
done    = {r['image'] for r in records}

for fname in get_all_images():
    if fname in done: continue
    concept, true_legs, bg = parse_filename(fname)
    img = Image.open(os.path.join(IMAGE_DIR, fname)).convert('RGB')

    conv = f"USER: <image>\n{PROMPT}\nASSISTANT:"
    inputs = processor(text=conv, images=img, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=10, do_sample=False)
    text = processor.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    pred = parse_legs(text)
    is_biased = int(pred != true_legs) if pred is not None else 1

    records.append({"image": fname, "concept": concept, "true_legs": true_legs,
                    "background": bg, "predicted": pred, "is_biased": is_biased,
                    "model": "LLaVA-1.5-7B", "hf_id": MODEL_ID})
    save_results(records, CSV_OUT)
    print(f"LLaVA-1.5 | {fname} | pred={pred} | true={true_legs} | bias={is_biased}")

del model, processor; free_gpu()
print(f"\n✅ LLaVA-1.5-7B done. Results: {CSV_OUT}")

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

LLaVA-1.5 | binturong_3legs_natural.png | pred=4 | true=3 | bias=1
LLaVA-1.5 | binturong_3legs_plain.png | pred=4 | true=3 | bias=1
LLaVA-1.5 | binturong_5legs_natural.png | pred=4 | true=5 | bias=1
LLaVA-1.5 | binturong_5legs_plain.png | pred=4 | true=5 | bias=1
LLaVA-1.5 | cat_3legs_natural.png | pred=4 | true=3 | bias=1
LLaVA-1.5 | cat_3legs_plain.png | pred=4 | true=3 | bias=1
LLaVA-1.5 | cat_5legs_natural.png | pred=4 | true=5 | bias=1
LLaVA-1.5 | cat_5legs_plain.png | pred=4 | true=5 | bias=1
LLaVA-1.5 | cheetah_3legs_natural.png | pred=4 | true=3 | bias=1
LLaVA-1.5 | cheetah_3legs_plain.png | pred=4 | true=3 | bias=1
LLaVA-1.5 | cheetah_5legs_natural.jpg | pred=4 | true=5 | bias=1
LLaVA-1.5 | cheetah_5legs_plain.png | pred=4 | true=5 | bias=1
LLaVA-1.5 | deer_3legs_natural.png | pred=4 | true=3 | bias=1
LLaVA-1.5 | deer_3legs_plain.png | pred=4 | true=3 | bias=1
LLaVA-1.5 | deer_5legs_natural.jpg | pred=4 | true=5 | bias=1
LLaVA-1.5 | deer_5legs_plain.png | pred=4 | true=5 | bia

---
## Model 3: LLaVA-v1.6-Mistral-7B

In [7]:
from transformers import LlavaNextForConditionalGeneration, LlavaNextProcessor, BitsAndBytesConfig

MODEL_ID = "llava-hf/llava-v1.6-mistral-7b-hf"
CSV_OUT  = os.path.join(RESULTS_DIR, "llava16_mistral_results.csv")

processor = LlavaNextProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)
model = LlavaNextForConditionalGeneration.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config,
            device_map="auto",
            token=HF_TOKEN)

records = list(pd.read_csv(CSV_OUT).to_dict('records')) if os.path.exists(CSV_OUT) else []
done    = {r['image'] for r in records}

for fname in get_all_images():
    if fname in done: continue
    concept, true_legs, bg = parse_filename(fname)
    img = Image.open(os.path.join(IMAGE_DIR, fname)).convert('RGB')

    conv = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": PROMPT}]}]
    prompt_text = processor.apply_chat_template(conv, add_generation_prompt=True)
    inputs = processor(text=prompt_text, images=img, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=10, do_sample=False)
    text = processor.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    pred = parse_legs(text)
    is_biased = int(pred != true_legs) if pred is not None else 1

    records.append({"image": fname, "concept": concept, "true_legs": true_legs,
                    "background": bg, "predicted": pred, "is_biased": is_biased,
                    "model": "LLaVA-v1.6-Mistral-7B", "hf_id": MODEL_ID})
    save_results(records, CSV_OUT)
    print(f"LLaVA-v1.6 | {fname} | pred={pred} | true={true_legs} | bias={is_biased}")

del model, processor; free_gpu()
print(f"\n✅ LLaVA-v1.6-Mistral done. Results: {CSV_OUT}")

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/687 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


LLaVA-v1.6 | binturong_3legs_natural.png | pred=4 | true=3 | bias=1


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

LLaVA-v1.6 | serval_3legs_natural.png | pred=4 | true=3 | bias=1


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


LLaVA-v1.6 | serval_3legs_plain.png | pred=4 | true=3 | bias=1


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


LLaVA-v1.6 | serval_5legs_natural.jpg | pred=4 | true=5 | bias=1
LLaVA-v1.6 | serval_5legs_plain.png | pred=4 | true=5 | bias=1
GPU freed. Current VRAM: 0.02 GB

✅ LLaVA-v1.6-Mistral done. Results: /content/drive/MyDrive/MS Research/PE/data/raw/results/llava16_mistral_results.csv


---
## Model 4: Phi-3.5-Vision-Instruct (4.2B)

In [7]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
import re
import os

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
CSV_OUT  = os.path.join(RESULTS_DIR, "qwen2vl_2b_results.csv")

model = Qwen2VLForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=256*28*28, max_pixels=512*28*28)

records = list(pd.read_csv(CSV_OUT).to_dict('records')) if os.path.exists(CSV_OUT) else []
done    = {r['image'] for r in records}

for fname in get_all_images():
    if fname in done: print(f"  skip {fname}"); continue
    concept, true_legs, bg = parse_filename(fname)
    img_path = os.path.join(IMAGE_DIR, fname)

    messages = [{"role": "user", "content": [
        {"type": "image", "image": f"file://{img_path}"},
        {"type": "text", "text": PROMPT}
    ]}]
    text_input = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text_input], images=image_inputs, padding=True, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=10, do_sample=False)
    text = processor.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    m = re.search(r'\d+', text)
    pred = int(m.group()) if m else None
    is_biased = int(pred != true_legs) if pred is not None else 1

    records.append({"image": fname, "concept": concept, "true_legs": true_legs,
                    "background": bg, "predicted": pred, "is_biased": is_biased,
                    "model": "Qwen2-VL-2B", "hf_id": MODEL_ID})
    save_results(records, CSV_OUT)
    print(f"Qwen2-VL | {fname} | raw='{text}' | pred={pred} | true={true_legs} | bias={is_biased}")

del model, processor; free_gpu()
print(f"\n✅ Qwen2-VL-2B done. Results: {CSV_OUT}")


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

  skip binturong_3legs_natural.png
  skip binturong_3legs_plain.png
  skip binturong_5legs_natural.png
  skip binturong_5legs_plain.png
  skip cat_3legs_natural.png
  skip cat_3legs_plain.png
  skip cat_5legs_natural.png
  skip cat_5legs_plain.png
  skip cheetah_3legs_natural.png
  skip cheetah_3legs_plain.png
  skip cheetah_5legs_natural.jpg
  skip cheetah_5legs_plain.png
  skip deer_3legs_natural.png
  skip deer_3legs_plain.png
  skip deer_5legs_natural.jpg
  skip deer_5legs_plain.png
  skip dog_3legs_natural.png
Qwen2-VL | dog_3legs_plain.png | raw='4' | pred=4 | true=3 | bias=1
Qwen2-VL | dog_5legs_natural.png | raw='4' | pred=4 | true=5 | bias=1
Qwen2-VL | dog_5legs_plain.png | raw='4' | pred=4 | true=5 | bias=1
Qwen2-VL | elephant_3legs_natural.png | raw='4' | pred=4 | true=3 | bias=1
Qwen2-VL | elephant_3legs_plain.png | raw='4' | pred=4 | true=3 | bias=1
Qwen2-VL | elephant_5legs_natural.png | raw='4' | pred=4 | true=5 | bias=1
Qwen2-VL | elephant_5legs_plain.png | raw='4' | pr

---
## Model 5: InternVL2_5-4B

In [9]:
from transformers import InstructBlipForConditionalGeneration, InstructBlipProcessor
import re

MODEL_ID = "Salesforce/instructblip-flan-t5-xl"
CSV_OUT  = os.path.join(RESULTS_DIR, "instructblip_results.csv")

processor = InstructBlipProcessor.from_pretrained(MODEL_ID)
model = InstructBlipForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16,  device_map="cuda").eval()

records = list(pd.read_csv(CSV_OUT).to_dict('records')) if os.path.exists(CSV_OUT) else []
done    = {r['image'] for r in records}

for fname in get_all_images():
    if fname in done: print(f"  skip {fname}"); continue
    concept, true_legs, bg = parse_filename(fname)
    img = Image.open(os.path.join(IMAGE_DIR, fname)).convert('RGB')

    inputs = processor(images=img, text="How many legs does this animal have?", return_tensors="pt").to("cuda", dtype=torch.bfloat16)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=10)
    text = processor.decode(out[0], skip_special_tokens=True).strip()
    m = re.search(r'\d+', text)
    pred = int(m.group()) if m else None
    is_biased = int(pred != true_legs) if pred is not None else 1

    records.append({"image": fname, "concept": concept, "true_legs": true_legs,
                    "background": bg, "predicted": pred, "is_biased": is_biased,
                    "model": "InstructBLIP-FlanT5XL", "hf_id": MODEL_ID})
    save_results(records, CSV_OUT)
    print(f"InstructBLIP | {fname} | raw='{text}' | pred={pred} | true={true_legs} | bias={is_biased}")

del model, processor; free_gpu()
print(f"\n✅ InstructBLIP done. Results: {CSV_OUT}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


processor_config.json:   0%|          | 0.00/75.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/439 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1363 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie language_model.shared.weight to language_model.lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

InstructBLIP | binturong_3legs_natural.png | raw='4' | pred=4 | true=3 | bias=1
InstructBLIP | binturong_3legs_plain.png | raw='4' | pred=4 | true=3 | bias=1
InstructBLIP | binturong_5legs_natural.png | raw='4' | pred=4 | true=5 | bias=1
InstructBLIP | binturong_5legs_plain.png | raw='4' | pred=4 | true=5 | bias=1
InstructBLIP | cat_3legs_natural.png | raw='4' | pred=4 | true=3 | bias=1
InstructBLIP | cat_3legs_plain.png | raw='4' | pred=4 | true=3 | bias=1
InstructBLIP | cat_5legs_natural.png | raw='4' | pred=4 | true=5 | bias=1
InstructBLIP | cat_5legs_plain.png | raw='4' | pred=4 | true=5 | bias=1
InstructBLIP | cheetah_3legs_natural.png | raw='4' | pred=4 | true=3 | bias=1
InstructBLIP | cheetah_3legs_plain.png | raw='4' | pred=4 | true=3 | bias=1
InstructBLIP | cheetah_5legs_natural.jpg | raw='4' | pred=4 | true=5 | bias=1
InstructBLIP | cheetah_5legs_plain.png | raw='4' | pred=4 | true=5 | bias=1
InstructBLIP | deer_3legs_natural.png | raw='4' | pred=4 | true=3 | bias=1
InstructB

--
## Model 6: Gemma-3-4B ⚠️ BF16 only — no FP16!

In [15]:
from transformers import AutoProcessor, AutoModelForCausalLM
import torch, re, os
import pandas as pd
from PIL import Image

MODEL_ID = "google/gemma-3-4b-it"
CSV_OUT  = os.path.join(RESULTS_DIR, "gemma3_4b_results.csv")

# CRITICAL: Must use bfloat16 — FP16 causes NaN overflow on this model
processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16,
                device_map="auto", token=HF_TOKEN).eval()

records = list(pd.read_csv(CSV_OUT).to_dict('records')) if os.path.exists(CSV_OUT) else []
done    = {r['image'] for r in records}

for fname in get_all_images():
    if fname in done: 
        print(f"  skip {fname}"); continue
    
    concept, true_legs, bg = parse_filename(fname)
    img = Image.open(os.path.join(IMAGE_DIR, fname)).convert('RGB')

    messages = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": PROMPT}]}]
    prompt_text = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    
    inputs = processor(text=prompt_text, images=img, return_tensors="pt").to(model.device, torch.bfloat16)

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=10, do_sample=False)
    
    text = processor.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    m = re.search(r'\d+', text)
    pred = int(m.group()) if m else None
    is_biased = int(pred != true_legs) if pred is not None else 1

    records.append({"image": fname, "concept": concept, "true_legs": true_legs,
                    "background": bg, "predicted": pred, "is_biased": is_biased,
                    "model": "Gemma-3-4B", "hf_id": MODEL_ID})
    save_results(records, CSV_OUT)
    print(f"Gemma-3 | {fname} | raw='{text}' | pred={pred} | true={true_legs} | bias={is_biased}")

del model, processor; free_gpu()
print(f"\n✅ Gemma-3-4B done. Results: {CSV_OUT}")


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Gemma-3 | binturong_3legs_natural.png | raw='4' | pred=4 | true=3 | bias=1
Gemma-3 | binturong_3legs_plain.png | raw='4' | pred=4 | true=3 | bias=1
Gemma-3 | binturong_5legs_natural.png | raw='4' | pred=4 | true=5 | bias=1
Gemma-3 | binturong_5legs_plain.png | raw='4' | pred=4 | true=5 | bias=1
Gemma-3 | cat_3legs_natural.png | raw='4' | pred=4 | true=3 | bias=1
Gemma-3 | cat_3legs_plain.png | raw='4' | pred=4 | true=3 | bias=1
Gemma-3 | cat_5legs_natural.png | raw='4' | pred=4 | true=5 | bias=1
Gemma-3 | cat_5legs_plain.png | raw='4' | pred=4 | true=5 | bias=1
Gemma-3 | cheetah_3legs_natural.png | raw='4' | pred=4 | true=3 | bias=1
Gemma-3 | cheetah_3legs_plain.png | raw='4' | pred=4 | true=3 | bias=1
Gemma-3 | cheetah_5legs_natural.jpg | raw='4' | pred=4 | true=5 | bias=1
Gemma-3 | cheetah_5legs_plain.png | raw='4' | pred=4 | true=5 | bias=1
Gemma-3 | deer_3legs_natural.png | raw='4' | pred=4 | true=3 | bias=1
Gemma-3 | deer_3legs_plain.png | raw='4' | pred=4 | true=3 | bias=1
Gemma-

---
## Model 7: Qwen2.5-VL-7B ⚠️ Pixel cap applied to avoid VRAM spike

In [17]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info
import torch, re, os
import pandas as pd
from PIL import Image

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
CSV_OUT  = os.path.join(RESULTS_DIR, "qwen25_vl_results.csv")

# Bulletproof 4-bit Config: Cuts VRAM in half and prevents silent kernel freezes
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# Cap max pixels slightly lower to guarantee it never spikes VRAM
MIN_PIX = 256  * 28 * 28
MAX_PIX = 768 * 28 * 28 

processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=MIN_PIX, max_pixels=MAX_PIX)

print("Starting 4-bit model load (this will take a few minutes)...")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    device_map="cuda",
    quantization_config=quant_config
).eval()
print(f"Model loaded successfully! Current VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

records = list(pd.read_csv(CSV_OUT).to_dict('records')) if os.path.exists(CSV_OUT) else []
done    = {r['image'] for r in records}

for fname in get_all_images():
    if fname in done: 
        print(f"  skip {fname}"); continue
        
    concept, true_legs, bg = parse_filename(fname)
    img_path = os.path.join(IMAGE_DIR, fname)

    messages = [{"role": "user", "content": [
        {"type": "image", "image": f"file://{img_path}", "min_pixels": MIN_PIX, "max_pixels": MAX_PIX},
        {"type": "text", "text": PROMPT}
    ]}]
    
    text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(messages)
    
    inputs = processor(text=[text_prompt], images=image_inputs, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=10, do_sample=False)
        
    text = processor.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    m = re.search(r'\d+', text)
    pred = int(m.group()) if m else None
    is_biased = int(pred != true_legs) if pred is not None else 1

    records.append({"image": fname, "concept": concept, "true_legs": true_legs,
                    "background": bg, "predicted": pred, "is_biased": is_biased,
                    "model": "Qwen2.5-VL-7B", "hf_id": MODEL_ID})
    save_results(records, CSV_OUT)
    print(f"Qwen2.5-VL | {fname} | raw='{text}' | pred={pred} | true={true_legs} | bias={is_biased}")

del model, processor; free_gpu()
print(f"\n✅ Qwen2.5-VL-7B done. Results: {CSV_OUT}")


Starting 4-bit model load (this will take a few minutes)...


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Model loaded successfully! Current VRAM: 5.95 GB


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Qwen2.5-VL | binturong_3legs_natural.png | raw='4' | pred=4 | true=3 | bias=1
Qwen2.5-VL | binturong_3legs_plain.png | raw='4' | pred=4 | true=3 | bias=1
Qwen2.5-VL | binturong_5legs_natural.png | raw='4' | pred=4 | true=5 | bias=1
Qwen2.5-VL | binturong_5legs_plain.png | raw='4' | pred=4 | true=5 | bias=1
Qwen2.5-VL | cat_3legs_natural.png | raw='4' | pred=4 | true=3 | bias=1
Qwen2.5-VL | cat_3legs_plain.png | raw='4' | pred=4 | true=3 | bias=1
Qwen2.5-VL | cat_5legs_natural.png | raw='4' | pred=4 | true=5 | bias=1
Qwen2.5-VL | cat_5legs_plain.png | raw='4' | pred=4 | true=5 | bias=1
Qwen2.5-VL | cheetah_3legs_natural.png | raw='4' | pred=4 | true=3 | bias=1
Qwen2.5-VL | cheetah_3legs_plain.png | raw='4' | pred=4 | true=3 | bias=1
Qwen2.5-VL | cheetah_5legs_natural.jpg | raw='4' | pred=4 | true=5 | bias=1
Qwen2.5-VL | cheetah_5legs_plain.png | raw='4' | pred=4 | true=5 | bias=1
Qwen2.5-VL | deer_3legs_natural.png | raw='4' | pred=4 | true=3 | bias=1
Qwen2.5-VL | deer_3legs_plain.png |

---
## Model 8: InternVL2-8B ⚠️ Borderline — close other browser tabs first

In [ ]:
# from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
# import torch, torchvision.transforms as T
# from torchvision.transforms.functional import InterpolationMode
# from PIL import Image
# import os, re
# import pandas as pd

# MODEL_ID = "OpenGVLab/InternVL2-8B"
# CSV_OUT  = os.path.join(RESULTS_DIR, "internvl2_8b_results.csv")

# IMAGENET_MEAN = (0.485, 0.456, 0.406)
# IMAGENET_STD = (0.229, 0.224, 0.225)

# def build_transform(input_size):
#     return T.Compose([
#         T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
#         T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
#         T.ToTensor(),
#         T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
#     ])

# transform = build_transform(input_size=448)

# print("Starting 4-bit loading (will take 3-4 minutes)...")
# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, token=HF_TOKEN)

# # 4-bit Config will save you from the VRAM crash!
# quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
# model = AutoModel.from_pretrained(
#     MODEL_ID, 
#     torch_dtype=torch.bfloat16,
#     trust_remote_code=True, 
#     device_map="auto",
#     quantization_config=quant_config,
#     token=HF_TOKEN
# ).eval()

# print(f"Model loaded correctly! Current VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# records = list(pd.read_csv(CSV_OUT).to_dict('records')) if os.path.exists(CSV_OUT) else []
# done    = {r['image'] for r in records}

# for fname in get_all_images():
#     if fname in done: 
#         print(f"  skip {fname}"); continue
        
#     concept, true_legs, bg = parse_filename(fname)
#     img = Image.open(os.path.join(IMAGE_DIR, fname)).convert('RGB')
    
#     # 4-bit model inference uses torch.bfloat16
#     pixel_values = transform(img).unsqueeze(0).to(model.device, dtype=torch.bfloat16)

#     response = model.chat(tokenizer, pixel_values, PROMPT, generation_config={"max_new_tokens": 10})
    
#     m = re.search(r'\d+', response)
#     pred = int(m.group()) if m else None
#     is_biased = int(pred != true_legs) if pred is not None else 1

#     records.append({"image": fname, "concept": concept, "true_legs": true_legs,
#                     "background": bg, "predicted": pred, "is_biased": is_biased,
#                     "model": "InternVL2-8B", "hf_id": MODEL_ID})
#     save_results(records, CSV_OUT)
#     print(f"InternVL2-8B | {fname} | raw='{response}' | pred={pred} | true={true_legs} | bias={is_biased}")

# del model, tokenizer; free_gpu()
# print(f"\n✅ InternVL2-8B done. Results: {CSV_OUT}")


Starting 4-bit loading (will take 3-4 minutes)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
Unrecognized keys in `rope_parameters` for 'rope_type'='dynamic': {'rope_theta'}
`torch_dtype` is deprecated! Use `dtype` instead!
Unrecognized keys in `rope_parameters` for 'rope_type'='dynamic': {'rope_theta'}
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


FlashAttention2 is not installed.


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

KeyboardInterrupt: 

---
## Model 9: Idefics3-8B-Llama3 ⚠️ Run last — borderline VRAM

In [8]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 81.9 MB/s eta 0:00:00:00:010:01
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [12]:
# Option 1: Latest stable (try this first)
!pip install -U transformers

# Option 2: If Option 1 still fails, install from HEAD
!pip install git+https://github.com/huggingface/transformers.git


  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-dqa3s5ov
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-dqa3s5ov
  Resolved https://github.com/huggingface/transformers.git to commit da37a4d95758f90a0b37253b33618b382c7ca722
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-5.3.0.dev0-py3-none-any.whl size=11186219 sha256=2d82437628954b29bfd7667265e77ba880c6fe4304f40f3a3a5c232f98e170e7
  Stored in directory: /tmp/pip-ephem-wheel-cache-58tjza3_/wheels/54/cb/3f/83103de5575c534436d6a4686686dead458238dfaf1147e98d
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 5.3.0
    Uninstalling transformers-5.3.0:
      Successfully uninstalled transformers-5.3.0


In [13]:
from transformers import (
    AutoProcessor,
    Idefics3ForConditionalGeneration,
    BitsAndBytesConfig,
)
from tqdm import tqdm
import torch, os, pandas as pd
from PIL import Image

# ── Config ──────────────────────────────────────────────────────────────────
MODEL_ID = "HuggingFaceM4/Idefics3-8B-Llama3"
CSV_OUT  = os.path.join(RESULTS_DIR, "idefics3_results.csv")

# ── 4-bit quantization to stay within 16 GB VRAM ────────────────────────────
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = Idefics3ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    device_map="auto",
    token=HF_TOKEN,
)
model.eval()
print(f"VRAM after load: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# ── Resume-safe result loading ───────────────────────────────────────────────
records = list(pd.read_csv(CSV_OUT).to_dict("records")) if os.path.exists(CSV_OUT) else []
done    = {r["image"] for r in records}
pending = [f for f in get_all_images() if f not in done]
print(f"Pending: {len(pending)} / {len(get_all_images())} images")

# ── Inference loop ───────────────────────────────────────────────────────────
for fname in tqdm(pending, desc="Idefics3"):
    try:
        concept, true_legs, bg = parse_filename(fname)
        img = Image.open(os.path.join(IMAGE_DIR, fname)).convert("RGB")

        messages = [{"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": PROMPT},
        ]}]
        prompt_text = processor.apply_chat_template(messages, add_generation_prompt=True)
        inputs = processor(text=prompt_text, images=[img], return_tensors="pt").to(model.device)

        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=10, do_sample=False)

        decoded = processor.decode(
            out[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        ).strip()
        pred      = parse_legs(decoded)
        is_biased = int(pred != true_legs) if pred is not None else 1

        records.append({
            "image": fname, "concept": concept, "true_legs": true_legs,
            "background": bg, "predicted": pred, "raw_output": decoded,
            "is_biased": is_biased, "model": "Idefics3-8B-Llama3", "hf_id": MODEL_ID,
        })
        save_results(records, CSV_OUT)

    except Exception as e:
        print(f"\n⚠️  Skipped {fname}: {e}")
        records.append({
            "image": fname, "concept": None, "true_legs": None,
            "background": None, "predicted": None, "raw_output": str(e),
            "is_biased": 1, "model": "Idefics3-8B-Llama3", "hf_id": MODEL_ID,
        })
        save_results(records, CSV_OUT)

# ── Cleanup ──────────────────────────────────────────────────────────────────
del model, processor
free_gpu()
print(f"\n✅ Idefics3-8B done — {len(records)} rows → {CSV_OUT}")


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

ImportError: cannot import name 'merge_with_config_defaults' from 'transformers.utils.generic' (/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py)

---
## Model 10: Llama-4-Scout via Groq API (No GPU — Remote Cloud)

In [7]:
import requests, base64, time

CSV_OUT = os.path.join(RESULTS_DIR, "llama4scout_results.csv")

def encode_image_b64(path):
    with open(path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')

records = list(pd.read_csv(CSV_OUT).to_dict('records')) if os.path.exists(CSV_OUT) else []
done    = {r['image'] for r in records}

for fname in get_all_images():
    if fname in done: continue
    concept, true_legs, bg = parse_filename(fname)
    img_path = os.path.join(IMAGE_DIR, fname)
    ext = Path(fname).suffix.lstrip('.')
    b64 = encode_image_b64(img_path)

    payload = {
        "model": "meta-llama/llama-4-scout-17b-16e-instruct",
        "messages": [{
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/{ext};base64,{b64}"}},
                {"type": "text", "text": PROMPT}
            ]
        }],
        "max_tokens": 10,
        "temperature": 0.0
    }
    headers = {"Authorization": f"Bearer {GROQ_API_KEY}", "Content-Type": "application/json"}

    try:
        r = requests.post("https://api.groq.com/openai/v1/chat/completions", json=payload, headers=headers, timeout=30)
        text = r.json()['choices'][0]['message']['content'] if r.status_code == 200 else ""
        if r.status_code == 429: print("Rate limited, sleeping 10s..."); time.sleep(10); continue
    except Exception as e:
        print(f"Error: {e}"); text = ""

    pred = parse_legs(text)
    is_biased = int(pred != true_legs) if pred is not None else 1
    records.append({"image": fname, "concept": concept, "true_legs": true_legs,
                    "background": bg, "predicted": pred, "is_biased": is_biased,
                    "model": "Llama-4-Scout-17B", "hf_id": "meta-llama/llama-4-scout-17b-16e-instruct"})
    save_results(records, CSV_OUT)
    print(f"Llama-4-Scout | {fname} | pred={pred} | true={true_legs} | bias={is_biased}")
    time.sleep(1)  # Respect rate limit

print(f"\n✅ Llama-4-Scout done. Results: {CSV_OUT}")

Llama-4-Scout | binturong_3legs_natural.png | pred=4 | true=3 | bias=1
Llama-4-Scout | binturong_3legs_plain.png | pred=4 | true=3 | bias=1
Llama-4-Scout | binturong_5legs_natural.png | pred=4 | true=5 | bias=1
Llama-4-Scout | binturong_5legs_plain.png | pred=4 | true=5 | bias=1
Llama-4-Scout | cat_3legs_natural.png | pred=4 | true=3 | bias=1
Llama-4-Scout | cat_3legs_plain.png | pred=4 | true=3 | bias=1
Llama-4-Scout | cat_5legs_natural.png | pred=4 | true=5 | bias=1
Llama-4-Scout | cat_5legs_plain.png | pred=4 | true=5 | bias=1
Llama-4-Scout | cheetah_3legs_natural.png | pred=4 | true=3 | bias=1
Llama-4-Scout | cheetah_3legs_plain.png | pred=4 | true=3 | bias=1
Llama-4-Scout | cheetah_5legs_natural.jpg | pred=4 | true=5 | bias=1
Llama-4-Scout | cheetah_5legs_plain.png | pred=8 | true=5 | bias=1
Llama-4-Scout | deer_3legs_natural.png | pred=4 | true=3 | bias=1
Llama-4-Scout | deer_3legs_plain.png | pred=4 | true=3 | bias=1
Llama-4-Scout | deer_5legs_natural.jpg | pred=4 | true=5 | bia

Llama-4-Maverick

In [9]:
# ── Groq Shared Setup ──────────────────────────────────────────────
import requests, base64, time, re, os
import pandas as pd
from pathlib import Path

def encode_b64(path):
    """Base64-encode an image file for Groq API upload."""
    with open(path, 'rb') as f:
        return base64.b64encode(f.read()).decode()

GROQ_HEADERS = {
    "Authorization": f"Bearer {GROQ_API_KEY}",
    "Content-Type": "application/json"
}

def groq_query(model_id, img_path, prompt, retries=3):
    """Send one image+prompt to Groq and return the text response."""
    ext = Path(img_path).suffix.lstrip('.')
    b64 = encode_b64(img_path)
    payload = {"model": model_id, "messages": [{"role": "user", "content": [
        {"type": "image_url", "image_url": {"url": f"data:image/{ext};base64,{b64}"}},
        {"type": "text", "text": prompt}
    ]}], "max_tokens": 10, "temperature": 0.0}
    for _ in range(retries):
        try:
            r = requests.post("https://api.groq.com/openai/v1/chat/completions",
                              json=payload, headers=GROQ_HEADERS, timeout=30)
            if r.status_code == 429:
                print("Rate limited, sleeping 10s..."); time.sleep(10); continue
            if r.status_code == 200:
                return r.json()['choices'][0]['message']['content']
        except Exception as e:
            print(f"Error: {e}")
    return ""

print("✅ Groq shared setup ready.")


✅ Groq shared setup ready.


In [10]:
import requests, base64, time, re, os, pandas as pd
from pathlib import Path

CSV_OUT = os.path.join(RESULTS_DIR, "llama4_scout_results.csv")
MODEL_ID = "meta-llama/llama-4-scout-17b-16e-instruct"

def encode_b64(path):
    with open(path, 'rb') as f: return base64.b64encode(f.read()).decode()

records = list(pd.read_csv(CSV_OUT).to_dict('records')) if os.path.exists(CSV_OUT) else []
done = {r['image'] for r in records}

for fname in get_all_images():
    if fname in done: print(f"  skip {fname}"); continue
    concept, true_legs, bg = parse_filename(fname)
    img_path = os.path.join(IMAGE_DIR, fname)
    b64 = encode_b64(img_path)
    ext = Path(fname).suffix.lstrip('.')

    payload = {"model": MODEL_ID, "messages": [{"role": "user", "content": [
        {"type": "image_url", "image_url": {"url": f"data:image/{ext};base64,{b64}"}},
        {"type": "text", "text": PROMPT}
    ]}], "max_tokens": 10, "temperature": 0.0}
    headers = {"Authorization": f"Bearer {GROQ_API_KEY}", "Content-Type": "application/json"}

    try:
        r = requests.post("https://api.groq.com/openai/v1/chat/completions", json=payload, headers=headers, timeout=30)
        if r.status_code == 429: print("Rate limited, sleeping 10s..."); time.sleep(10); continue
        text = r.json()['choices'][0]['message']['content'] if r.status_code == 200 else ""
    except Exception as e: print(f"Error: {e}"); text = ""

    m = re.search(r'\d+', text)
    pred = int(m.group()) if m else None
    is_biased = int(pred != true_legs) if pred is not None else 1
    records.append({"image": fname, "concept": concept, "true_legs": true_legs,
                    "background": bg, "predicted": pred, "is_biased": is_biased,
                    "model": "Llama-4-Scout-17B", "hf_id": MODEL_ID})
    save_results(records, CSV_OUT)
    print(f"Scout | {fname} | raw='{text}' | pred={pred} | true={true_legs} | bias={is_biased}")
    time.sleep(1)

print(f"\n✅ Llama-4-Scout done. Results: {CSV_OUT}")


Scout | binturong_3legs_natural.png | raw='4.' | pred=4 | true=3 | bias=1
Scout | binturong_3legs_plain.png | raw='4.' | pred=4 | true=3 | bias=1
Scout | binturong_5legs_natural.png | raw='Four.' | pred=None | true=5 | bias=1
Scout | binturong_5legs_plain.png | raw='4.' | pred=4 | true=5 | bias=1
Scout | cat_3legs_natural.png | raw='4.' | pred=4 | true=3 | bias=1
Scout | cat_3legs_plain.png | raw='4.' | pred=4 | true=3 | bias=1
Scout | cat_5legs_natural.png | raw='4.' | pred=4 | true=5 | bias=1
Scout | cat_5legs_plain.png | raw='4.' | pred=4 | true=5 | bias=1
Scout | cheetah_3legs_natural.png | raw='4.' | pred=4 | true=3 | bias=1
Scout | cheetah_3legs_plain.png | raw='4.' | pred=4 | true=3 | bias=1
Scout | cheetah_5legs_natural.jpg | raw='4.' | pred=4 | true=5 | bias=1
Scout | cheetah_5legs_plain.png | raw='8.' | pred=8 | true=5 | bias=1
Scout | deer_3legs_natural.png | raw='4.' | pred=4 | true=3 | bias=1
Scout | deer_3legs_plain.png | raw='The image shows a deer walking, and it has' |

Gemma 4

In [6]:
# ── Install / upgrade transformers for Gemma 4 support ──
!pip install -q -U transformers accelerate pillow

In [7]:
!pip install -q -U pillow


In [8]:
# ── CELL: Load Gemma 4 E2B (run this BEFORE the loop) ──
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

MODEL_ID = "google/gemma-4-E2B-it"

print("Loading Gemma 4 E2B...")
processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=HF_TOKEN
).eval()

print(f"✅ Loaded! VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"processor type: {type(processor)}")   # confirms it's defined


Loading Gemma 4 E2B...


processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

✅ Loaded! VRAM: 10.25 GB
processor type: <class 'transformers.models.gemma4.processing_gemma4.Gemma4Processor'>


In [10]:
CSV_OUT = os.path.join(RESULTS_DIR, "gemma4_e2b_results.csv")

records = list(pd.read_csv(CSV_OUT).to_dict('records')) if os.path.exists(CSV_OUT) else []
done    = {r['image'] for r in records}

for fname in get_all_images():
    if fname in done:
        print(f"  skip {fname}"); continue

    concept, true_legs, bg = parse_filename(fname)
    img = Image.open(os.path.join(IMAGE_DIR, fname)).convert("RGB")

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": img},
            {"type": "text",  "text": PROMPT}
        ]
    }]

    # ── FIX: tokenize=False → get string, then processor() → get dict ──
    prompt_text = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False
    )
    inputs = processor(text=prompt_text, images=img, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=10, do_sample=False)

    response = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    m = re.search(r'\d+', response)
    pred = int(m.group()) if m else None
    is_biased = int(pred != true_legs) if pred is not None else 1

    records.append({"image": fname, "concept": concept, "true_legs": true_legs,
                    "background": bg, "predicted": pred, "is_biased": is_biased,
                    "model": "Gemma4-E2B", "hf_id": MODEL_ID})
    save_results(records, CSV_OUT)
    print(f"Gemma4-E2B | {fname} | raw='{response}' | pred={pred} | true={true_legs} | bias={is_biased}")

del model, processor; free_gpu()
print(f"\n✅ Gemma4-E2B done. Results: {CSV_OUT}")


The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Gemma4-E2B | binturong_3legs_natural.png | raw='4' | pred=4 | true=3 | bias=1
Gemma4-E2B | binturong_3legs_plain.png | raw='4' | pred=4 | true=3 | bias=1
Gemma4-E2B | binturong_5legs_natural.png | raw='4' | pred=4 | true=5 | bias=1
Gemma4-E2B | binturong_5legs_plain.png | raw='4' | pred=4 | true=5 | bias=1
Gemma4-E2B | cat_3legs_natural.png | raw='4' | pred=4 | true=3 | bias=1
Gemma4-E2B | cat_3legs_plain.png | raw='4' | pred=4 | true=3 | bias=1
Gemma4-E2B | cat_5legs_natural.png | raw='4' | pred=4 | true=5 | bias=1
Gemma4-E2B | cat_5legs_plain.png | raw='4' | pred=4 | true=5 | bias=1
Gemma4-E2B | cheetah_3legs_natural.png | raw='4' | pred=4 | true=3 | bias=1
Gemma4-E2B | cheetah_3legs_plain.png | raw='4' | pred=4 | true=3 | bias=1
Gemma4-E2B | cheetah_5legs_natural.jpg | raw='4' | pred=4 | true=5 | bias=1
Gemma4-E2B | cheetah_5legs_plain.png | raw='4' | pred=4 | true=5 | bias=1
Gemma4-E2B | deer_3legs_natural.png | raw='4' | pred=4 | true=3 | bias=1
Gemma4-E2B | deer_3legs_plain.png |

In [14]:
# ── LOAD MODEL + TEST OUTPUT (all-in-one) ──
from transformers import AutoProcessor, AutoModelForImageTextToText
from PIL import Image
import torch, os

MODEL_ID = "google/gemma-4-E2B-it"   # or "google/gemma-3-4b-it" whichever worked

processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto", token=HF_TOKEN
).eval()
print(f"✅ Loaded! VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# ── Quick test ──
test_img = get_all_images()[0]
img = Image.open(os.path.join(IMAGE_DIR, test_img)).convert("RGB")

messages = [{"role": "user", "content": [
    {"type": "image", "image": img},
    {"type": "text",  "text": PROMPT}
]}]

prompt_text = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
inputs = processor(text=prompt_text, images=img, return_tensors="pt").to(model.device)

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=50, do_sample=False)

response = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(f"\nImage   : {test_img}")
print(f"Prompt  : {PROMPT}")
print(f"Answer  : '{response}'")


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

✅ Loaded! VRAM: 10.26 GB

Image   : binturong_3legs_natural.png
Prompt  : How many legs does this animal exactly have in this image? Answer with just a single number integer, nothing else.
Answer  : '4'


In [17]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

MODEL_ID = "google/gemma-4-E4B-it"   # ← only change

processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto", token=HF_TOKEN
).eval()
print(f"✅ Loaded! VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")


processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Current model requires 6192 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

✅ Loaded! VRAM: 0.02 GB


In [18]:
import re, pandas as pd
from PIL import Image

CSV_OUT = os.path.join(RESULTS_DIR, "gemma4_e4b_results.csv")   # ← separate CSV

records = list(pd.read_csv(CSV_OUT).to_dict('records')) if os.path.exists(CSV_OUT) else []
done    = {r['image'] for r in records}

for fname in get_all_images():
    if fname in done:
        print(f"  skip {fname}"); continue

    concept, true_legs, bg = parse_filename(fname)
    img = Image.open(os.path.join(IMAGE_DIR, fname)).convert("RGB")

    messages = [{"role": "user", "content": [
        {"type": "image", "image": img},
        {"type": "text",  "text": PROMPT}
    ]}]

    prompt_text = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = processor(text=prompt_text, images=img, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=10, do_sample=False)

    response = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    m = re.search(r'\d+', response)
    pred = int(m.group()) if m else None
    is_biased = int(pred != true_legs) if pred is not None else 1

    records.append({"image": fname, "concept": concept, "true_legs": true_legs,
                    "background": bg, "predicted": pred, "is_biased": is_biased,
                    "model": "Gemma4-E4B", "hf_id": MODEL_ID})
    save_results(records, CSV_OUT)
    print(f"Gemma4-E4B | {fname} | raw='{response}' | pred={pred} | true={true_legs} | bias={is_biased}")

del model, processor; free_gpu()
print(f"\n✅ Gemma4-E4B done. Results: {CSV_OUT}")


Gemma4-E4B | binturong_3legs_natural.png | raw='4' | pred=4 | true=3 | bias=1
Gemma4-E4B | binturong_3legs_plain.png | raw='4' | pred=4 | true=3 | bias=1
Gemma4-E4B | binturong_5legs_natural.png | raw='4' | pred=4 | true=5 | bias=1
Gemma4-E4B | binturong_5legs_plain.png | raw='4' | pred=4 | true=5 | bias=1
Gemma4-E4B | cat_3legs_natural.png | raw='4' | pred=4 | true=3 | bias=1
Gemma4-E4B | cat_3legs_plain.png | raw='4' | pred=4 | true=3 | bias=1
Gemma4-E4B | cat_5legs_natural.png | raw='4' | pred=4 | true=5 | bias=1
Gemma4-E4B | cat_5legs_plain.png | raw='4' | pred=4 | true=5 | bias=1
Gemma4-E4B | cheetah_3legs_natural.png | raw='4' | pred=4 | true=3 | bias=1
Gemma4-E4B | cheetah_3legs_plain.png | raw='4' | pred=4 | true=3 | bias=1
Gemma4-E4B | cheetah_5legs_natural.jpg | raw='4' | pred=4 | true=5 | bias=1
Gemma4-E4B | cheetah_5legs_plain.png | raw='4' | pred=4 | true=5 | bias=1
Gemma4-E4B | deer_3legs_natural.png | raw='4' | pred=4 | true=3 | bias=1
Gemma4-E4B | deer_3legs_plain.png |

In [19]:
!pip install groq

In [20]:
# ── Llama 4 Scout via Groq API (Full quality, zero VRAM) ──────────────────
from groq import Groq
import base64, re, time, os, pandas as pd

MODEL_ID = "meta-llama/llama-4-scout-17b-16e-instruct"
CSV_OUT  = os.path.join(RESULTS_DIR, "llama4_scout_results_new.csv")

client = Groq(api_key=GROQ_API_KEY)

def groq_vlm_infer(image_path, prompt, retries=3):
    """Send image + prompt to Llama 4 Scout via Groq. Returns raw text response."""
    ext  = image_path.split(".")[-1].lower()
    mime = "image/jpeg" if ext in ("jpg", "jpeg") else "image/png"
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()

    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL_ID,
                messages=[{"role": "user", "content": [
                    {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}},
                    {"type": "text",      "text": prompt}
                ]}],
                max_tokens=10,
                temperature=0.0   # greedy = deterministic, controlled
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            print(f"  [attempt {attempt+1}] Error: {e}")
            time.sleep(5 * (attempt + 1))
    return None

# ── Resume-safe loop ──────────────────────────────────────────────────────
records = list(pd.read_csv(CSV_OUT).to_dict('records')) if os.path.exists(CSV_OUT) else []
done    = {r['image'] for r in records}

for fname in get_all_images():
    if fname in done:
        print(f"  skip {fname}"); continue

    concept, true_legs, bg = parse_filename(fname)
    img_path = os.path.join(IMAGE_DIR, fname)

    response = groq_vlm_infer(img_path, PROMPT)
    if response is None:
        print(f"  ❌ FAILED: {fname} — skipping"); continue

    m = re.search(r'\d+', response)
    pred = int(m.group()) if m else None
    is_biased = int(pred != true_legs) if pred is not None else 1

    records.append({"image": fname, "concept": concept, "true_legs": true_legs,
                    "background": bg, "predicted": pred, "is_biased": is_biased,
                    "model": "Llama4-Scout", "hf_id": MODEL_ID})
    save_results(records, CSV_OUT)
    print(f"Llama4-Scout | {fname} | raw='{response}' | pred={pred} | true={true_legs} | bias={is_biased}")

    time.sleep(1.0)   # polite rate limit — Groq free tier ~30 req/min

print(f"\n✅ Llama4-Scout done. Results: {CSV_OUT}")


Llama4-Scout | binturong_3legs_natural.png | raw='4.' | pred=4 | true=3 | bias=1
Llama4-Scout | binturong_3legs_plain.png | raw='4.' | pred=4 | true=3 | bias=1
Llama4-Scout | binturong_5legs_natural.png | raw='4.' | pred=4 | true=5 | bias=1
Llama4-Scout | binturong_5legs_plain.png | raw='4.' | pred=4 | true=5 | bias=1
Llama4-Scout | cat_3legs_natural.png | raw='4.' | pred=4 | true=3 | bias=1
Llama4-Scout | cat_3legs_plain.png | raw='4.' | pred=4 | true=3 | bias=1
Llama4-Scout | cat_5legs_natural.png | raw='4.' | pred=4 | true=5 | bias=1
Llama4-Scout | cat_5legs_plain.png | raw='4.' | pred=4 | true=5 | bias=1
Llama4-Scout | cheetah_3legs_natural.png | raw='4' | pred=4 | true=3 | bias=1
Llama4-Scout | cheetah_3legs_plain.png | raw='4.' | pred=4 | true=3 | bias=1
Llama4-Scout | cheetah_5legs_natural.jpg | raw='4.' | pred=4 | true=5 | bias=1
Llama4-Scout | cheetah_5legs_plain.png | raw='8.' | pred=8 | true=5 | bias=1
Llama4-Scout | deer_3legs_natural.png | raw='4.' | pred=4 | true=3 | bias